# Qualitative figures (Figs 4-5 regeneration)

Reproduces the side-by-side comparison panels from the paper:
- Figure 4: MedFocus vs. baselines on three representative samples.
- Figure 5: token-level concept attribution for a reasoning example.

Requires that you have already run `scripts/run_attribution.py` for at least
MedFocus and a few baselines.

In [ ]:
import json
import sys, os
sys.path.insert(0, '..')
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

from medfocus.config import load_config
from medfocus.data.io import resize_pad, safe_open_image
from medfocus.data.imagenome import load_imagenome
from medfocus.data.medground_bench import attach_source_metadata
from medfocus.attribution.visualization import overlay_boxes

cfg = load_config('../configs')
image_size = cfg.datasets.image['resize_width']

In [ ]:
RESULTS_DIR = Path('../results')   # adjust to where run_attribution.py wrote its JSON
methods = ['medfocus', 'gradcam', 'gradcampp', 'attention_rollout']
by_method = {m: json.load(open(RESULTS_DIR / 'direct' / 'medgemma1_5_4b' / f'{m}.json')) for m in methods}
len(by_method['medfocus'])

In [ ]:
imagenome = load_imagenome(cfg.datasets.data_root, suffixes=cfg.datasets.question_suffixes)

# Pick three representative samples (one per dataset). Replace with your own choices.
datasets = ['imagenome', 'vindr_cxr', 'padchest_gr']
rows = []
for d in datasets:
    rec = next(r for r in by_method['medfocus'] if r['dataset'] == d)
    rows.append(rec)

fig, axes = plt.subplots(len(rows), 1 + len(methods), figsize=(4 * (1 + len(methods)), 4 * len(rows)))
for i, rec in enumerate(rows):
    record_with_meta = attach_source_metadata([rec], {'imagenome': imagenome})[0]
    img = resize_pad(safe_open_image(record_with_meta['imgpath']), image_size)
    axes[i, 0].imshow(overlay_boxes(img, record_with_meta['locations'], color='red'))
    axes[i, 0].set_title(f"{rec['dataset']} — GT")
    axes[i, 0].axis('off')
    for j, m in enumerate(methods, start=1):
        match = next(
            r for r in by_method[m]
            if r['dataset'] == rec['dataset'] and r['index'] == rec['index']
        )
        axes[i, j].imshow(overlay_boxes(img, match['pred_boxes'], color='yellow'))
        axes[i, j].set_title(m)
        axes[i, j].axis('off')
plt.tight_layout()
plt.savefig('method_comparison.pdf', bbox_inches='tight')
plt.show()